# 🛡️ TrustShield AI — Master Deepfake Detection & Triage Pipeline
**Course:** MPBA G514 — Deep Learning for Business  
**Domain:** Digital KYC / Automated Financial Fraud Prevention  
**Architecture:** Pretrained EfficientNet-B0 with Two-Stage Transfer Learning  

---
### Core Pipeline Architecture:
1. **Hardcoded Ground Truth:** `Real = 0` (Authentic), `Manipulated = 1` (Fraud).
2. **Two-Stage Fine-Tuning:**
   * *Stage 1 (Warmup):* 3 epochs training only the custom classification head (`lr=1e-3`) with backbone frozen.
   * *Stage 2 (Fine-Tuning):* 10 epochs training all layers with `CosineAnnealingLR` (`lr=1e-4`).
3. **Evaluations:** Clean Test Benchmark (Accuracy, F1, ROC-AUC, Confusion Matrix).
4. **Visual Interpretability:** Native Grad-CAM saliency heatmaps on final convolutional block.
5. **Transmission Stress-Testing:** 4-tier JPEG compression degradation sweep ($QF \in [100, 90, 75, 50, 30]$).
6. **MBA Managerial Analytics:** Cost-matrix threshold optimization ($\tau^*$) & Three-Tier Triage P&L modeling.

## 1. Hardware Check & Imports

In [ ]:
# Check GPU
!nvidia-smi

import os
import io
import time
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using compute device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Ingest Dataset (`data_processed.zip`)
Upload `data_processed.zip` (58 MB) to the Colab files pane.

In [ ]:
zip_path = 'data_processed.zip'
extract_dir = './data/processed'

if os.path.exists(zip_path):
    print(f"Extracting {zip_path}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    print("Extraction complete!")
else:
    print("⚠️ Please upload data_processed.zip into the Colab Files tab.")

# Verify counts
for split in ['train', 'val', 'test']:
    r_count = len(os.listdir(os.path.join(extract_dir, split, 'real')))
    m_count = len(os.listdir(os.path.join(extract_dir, split, 'manipulated')))
    print(f"Split: {split.upper():5s} | Real: {r_count:4d} | Manipulated: {m_count:4d} | Total: {r_count+m_count:4d}")

## 3. Explicit Dataset Mapping (Guarantees Real = 0, Manipulated = 1)
Eliminates any possibility of class inversion bugs.

In [ ]:
class ExplicitFaceDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.samples = []
        self.transform = transform
        
        real_dir = os.path.join(root_dir, 'real')
        fake_dir = os.path.join(root_dir, 'manipulated')
        
        # Explicitly map: Real = 0 (Authentic), Manipulated = 1 (Fraud)
        if os.path.exists(real_dir):
            for f in sorted(os.listdir(real_dir)):
                if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                    self.samples.append((os.path.join(real_dir, f), 0))
                    
        if os.path.exists(fake_dir):
            for f in sorted(os.listdir(fake_dir)):
                if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                    self.samples.append((os.path.join(fake_dir, f), 1))
                    
    def __len__(self):
        return len(self.samples)
        
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.82, 1.0), ratio=(0.95, 1.05)), # Mitigates background bias
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.2), value='random') # Penalizes background memorization
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = ExplicitFaceDataset('./data/processed/train', transform=train_transform)
val_dataset = ExplicitFaceDataset('./data/processed/val', transform=eval_transform)
test_dataset = ExplicitFaceDataset('./data/processed/test', transform=eval_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f"Verified explicit classes: Authentic Real = 0 | Synthetic Fake = 1")


## 4. Model Architecture: EfficientNet-B0 Transfer Learning

In [ ]:
def build_efficientnet():
    weights = models.EfficientNet_B0_Weights.DEFAULT
    model = models.efficientnet_b0(weights=weights)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3, inplace=True),
        nn.Linear(in_features, 1)  # Single raw logit: higher value -> higher probability of Fake (1)
    )
    return model

model = build_efficientnet().to(device)
print(f"Model built successfully. Classification head configured for binary output.")

## 5. Two-Stage Training Pipeline (Warmup + Fine-Tuning)

In [ ]:
criterion = nn.BCEWithLogitsLoss()
best_val_loss = float('inf')
os.makedirs('output', exist_ok=True)
best_model_path = 'output/best_model.pth'

# =========================================================
# STAGE 1: WARMUP (3 Epochs with Frozen Backbone)
# =========================================================
print("=" * 60)
print("STAGE 1: Training Custom Classification Head (Backbone Frozen)")
print("=" * 60)
for p in model.features.parameters():
    p.requires_grad = False

optimizer_head = torch.optim.AdamW(model.classifier.parameters(), lr=1e-3, weight_decay=1e-2)

for epoch in range(1, 4):
    model.train()
    tr_loss, tr_corr, tr_tot = 0.0, 0, 0
    for imgs, labels in train_loader:
        imgs = imgs.to(device)
        labels = labels.float().unsqueeze(1).to(device)
        optimizer_head.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_head.step()
        
        tr_loss += loss.item() * imgs.size(0)
        preds = (torch.sigmoid(outputs) >= 0.5).float()
        tr_corr += (preds == labels).sum().item()
        tr_tot += labels.size(0)
        
    model.eval()
    val_loss, val_corr, val_tot = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.to(device)
            labels = labels.float().unsqueeze(1).to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * imgs.size(0)
            preds = (torch.sigmoid(outputs) >= 0.5).float()
            val_corr += (preds == labels).sum().item()
            val_tot += labels.size(0)
            
    print(f"Warmup [{epoch}/3] Train Loss: {tr_loss/tr_tot:.4f}, Acc: {tr_corr/tr_tot*100:.2f}% | "
          f"Val Loss: {val_loss/val_tot:.4f}, Acc: {val_corr/val_tot*100:.2f}%")

# =========================================================
# STAGE 2: FULL FINE-TUNING (10 Epochs with Unfrozen Backbone)
# =========================================================
print("\n" + "=" * 60)
print("STAGE 2: Full Fine-Tuning (All Layers Unfrozen)")
print("=" * 60)
for p in model.features.parameters():
    p.requires_grad = True

optimizer_full = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_full, T_max=10)

for epoch in range(1, 11):
    model.train()
    tr_loss, tr_corr, tr_tot = 0.0, 0, 0
    for imgs, labels in train_loader:
        imgs = imgs.to(device)
        labels = labels.float().unsqueeze(1).to(device)
        optimizer_full.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_full.step()
        
        tr_loss += loss.item() * imgs.size(0)
        preds = (torch.sigmoid(outputs) >= 0.5).float()
        tr_corr += (preds == labels).sum().item()
        tr_tot += labels.size(0)
        
    scheduler.step()
    
    model.eval()
    val_loss, val_corr, val_tot = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.to(device)
            labels = labels.float().unsqueeze(1).to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * imgs.size(0)
            preds = (torch.sigmoid(outputs) >= 0.5).float()
            val_corr += (preds == labels).sum().item()
            val_tot += labels.size(0)
            
    ep_val_loss = val_loss / val_tot
    ep_val_acc = val_corr / val_tot
    print(f"FineTune [{epoch:02d}/10] Train Loss: {tr_loss/tr_tot:.4f}, Acc: {tr_corr/tr_tot*100:.2f}% | "
          f"Val Loss: {ep_val_loss:.4f}, Acc: {ep_val_acc*100:.2f}%")
          
    if ep_val_loss < best_val_loss:
        best_val_loss = ep_val_loss
        torch.save(model.state_dict(), best_model_path)
        print(f"  ⭐ Saved new best checkpoint ({ep_val_loss:.4f})")

## 6. Held-Out Clean Test Set Evaluation

In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load('output/best_model.pth'))
model.eval()

y_true, y_probs = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        probs = torch.sigmoid(outputs).cpu().numpy().flatten()
        y_probs.extend(probs)
        y_true.extend(labels.numpy().flatten())

y_true = np.array(y_true)
y_probs = np.array(y_probs)
y_pred = (y_probs >= 0.5).astype(int)

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred)
rec = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_probs)

print("=" * 55)
print("HELD-OUT CLEAN TEST EVALUATION (τ = 0.50):")
print(f"  Accuracy:               {acc * 100:.2f}%")
print(f"  Precision (Fake):       {prec * 100:.2f}%")
print(f"  Recall (Catch Rate):    {rec * 100:.2f}%")
print(f"  F1-Score:               {f1 * 100:.2f}%")
print(f"  ROC-AUC:                {auc:.4f}")
print("=" * 55)

# Export predictions for dashboard and inspection
df_preds = pd.DataFrame({'y_true': y_true, 'y_prob': y_probs, 'y_pred': y_pred})
os.makedirs('output', exist_ok=True)
df_preds.to_csv('output/test_predictions.csv', index=False)
print("Saved test_predictions.csv successfully.")

### Confusion Matrix & ROC Visuals

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
axes[0].matshow(cm, cmap='Blues', alpha=0.85)
for (i, j), z in np.ndenumerate(cm):
    axes[0].text(j, i, f'{z:,}', ha='center', va='center', fontsize=14, fontweight='bold')
axes[0].set_xticks([0, 1])
axes[0].set_yticks([0, 1])
axes[0].set_xticklabels(['Authentic (0)', 'Manipulated (1)'])
axes[0].set_yticklabels(['Authentic (0)', 'Manipulated (1)'])
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')
axes[0].set_title('Test Set Confusion Matrix', pad=15)

# ROC Curve
fpr, tpr, _ = roc_curve(y_true, y_probs)
axes[1].plot(fpr, tpr, color='#0284c7', linewidth=2.5, label=f'ROC Curve (AUC = {auc:.4f})')
axes[1].plot([0, 1], [0, 1], color='gray', linestyle='--')
axes[1].set_xlabel('False Positive Rate (FPR)')
axes[1].set_ylabel('True Positive Rate (TPR)')
axes[1].set_title('Receiver Operating Characteristic')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Explainability via Native Grad-CAM

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self.target_layer.register_forward_hook(self._save_activation)
        self.target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output

    def _save_gradient(self, module, grad_in, grad_out):
        self.gradients = grad_out[0]

    def __call__(self, x):
        self.model.zero_grad()
        output = self.model(x)
        output.backward(gradient=torch.ones_like(output))
        weights = torch.mean(self.gradients, dim=(2, 3), keepdim=True)
        cam = torch.sum(weights * self.activations, dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = cam - cam.min()
        if cam.max() > 0:
            cam = cam / cam.max()
        cam = F.interpolate(cam, size=x.shape[2:], mode='bilinear', align_corners=False)
        return cam.squeeze().detach().cpu().numpy(), torch.sigmoid(output).item()

grad_cam = GradCAM(model, model.features[-1])

# Visualize 4 representative samples
sample_indices = [0, 5, 225, 230] # 2 real, 2 fake
fig, axes = plt.subplots(len(sample_indices), 2, figsize=(7, 3.2 * len(sample_indices)))

for idx, s_idx in enumerate(sample_indices):
    img_path, true_label = test_dataset.samples[s_idx]
    raw_img = Image.open(img_path).convert('RGB').resize((224, 224))
    t_img = eval_transform(raw_img).unsqueeze(0).to(device)
    
    cam_map, score = grad_cam(t_img)
    label_str = 'Manipulated (Fake)' if true_label == 1 else 'Authentic (Real)'
    
    axes[idx, 0].imshow(raw_img)
    axes[idx, 0].set_title(f"True: {label_str} | Pred P(Fake): {score*100:.1f}%")
    axes[idx, 0].axis('off')
    
    axes[idx, 1].imshow(raw_img)
    axes[idx, 1].imshow(cam_map, cmap='inferno', alpha=0.55)
    axes[idx, 1].set_title("Grad-CAM Attention Heatmap")
    axes[idx, 1].axis('off')

plt.tight_layout()
plt.show()

## 8. Transmission Robustness Stress-Testing (JPEG Sweep)

In [ ]:
class JPEGSweepDataset(Dataset):
    def __init__(self, samples, quality, transform):
        self.samples = samples
        self.quality = quality
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.quality < 100:
            buf = io.BytesIO()
            img.save(buf, format='JPEG', quality=self.quality)
            buf.seek(0)
            img = Image.open(buf).convert('RGB')
        return self.transform(img), label

q_factors = [100, 90, 75, 50, 30]
sweep_accs, sweep_recs = [], []

print("Evaluating JPEG Compression Sweep on Test Partition...")
for q in q_factors:
    ds = JPEGSweepDataset(test_dataset.samples, quality=q, transform=eval_transform)
    loader = DataLoader(ds, batch_size=32, shuffle=False)
    
    q_true, q_probs = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            q_probs.extend(torch.sigmoid(outputs).cpu().numpy().flatten())
            q_true.extend(labels.numpy().flatten())
            
    q_pred = (np.array(q_probs) >= 0.5).astype(int)
    q_acc = accuracy_score(q_true, q_pred)
    q_rec = recall_score(q_true, q_pred)
    sweep_accs.append(q_acc * 100)
    sweep_recs.append(q_rec * 100)
    print(f"  QF = {q:3d} | Test Accuracy: {q_acc*100:.2f}% | Fraud Recall: {q_rec*100:.2f}%")

# Plot degradation curve
plt.figure(figsize=(7.5, 4))
plt.plot(q_factors, sweep_accs, marker='o', linewidth=2.5, color='#0284c7', label='Overall Accuracy (%)')
plt.plot(q_factors, sweep_recs, marker='s', linewidth=2.5, color='#e11d48', linestyle='--', label='Fraud Recall / Catch Rate (%)')
plt.axvline(x=70, color='gray', linestyle=':', label='Recommended Mobile Capture Floor (QF=70)')
plt.xlabel('JPEG Quality Factor (QF)')
plt.ylabel('Performance Metric (%)')
plt.title('Vulnerability Analysis: Model Degradation Under Compression')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 9. MBA Managerial Decision Analytics & Threshold Optimization

In [ ]:
MONTHLY_APPLICANTS = 100000
MANUAL_REVIEW_COST = 6.00      # $ per ticket
FRAUD_LOSS_PER_FN = 2500.00    # $ per missed fake

taus = np.linspace(0.02, 0.98, 100)
total_costs = []

for tau in taus:
    # FP: Authentic (0) flagged as Fake (>= tau)
    fp = np.sum((y_true == 0) & (y_probs >= tau))
    # FN: Fake (1) passed as Authentic (< tau)
    fn = np.sum((y_true == 1) & (y_probs < tau))
    
    fp_rate = fp / len(y_true)
    fn_rate = fn / len(y_true)
    
    exp_cost = (fn_rate * MONTHLY_APPLICANTS * FRAUD_LOSS_PER_FN) + \
               (fp_rate * MONTHLY_APPLICANTS * MANUAL_REVIEW_COST)
    total_costs.append(exp_cost)

best_idx = np.argmin(total_costs)
opt_tau = taus[best_idx]
min_loss = total_costs[best_idx]

print("=" * 55)
print(f"OPTIMAL FINANCIAL THRESHOLD: τ* = {opt_tau:.3f}")
print(f"Min Expected Loss at τ*:      ${min_loss:,.2f} / month")
print("=" * 55)

plt.figure(figsize=(7.5, 4))
plt.plot(taus, [c / 1000 for c in total_costs], color='#10b981', linewidth=2.5, label='Total Expected Enterprise Loss')
plt.axvline(x=opt_tau, color='#e11d48', linestyle='--', label=f'Optimal τ* = {opt_tau:.2f}')
plt.scatter([opt_tau], [min_loss / 1000], color='#e11d48', s=80, zorder=5)
plt.xlabel('Decision Threshold (τ)')
plt.ylabel('Monthly Cost ($ in Thousands)')
plt.title('Enterprise Cost-Matrix Optimization Curve')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

### Executive Scorecard

In [ ]:
tau_low = 0.20
tau_high = 0.75

green_count = np.sum(y_probs < tau_low)
yellow_count = np.sum((y_probs >= tau_low) & (y_probs <= tau_high))
red_count = np.sum(y_probs > tau_high)
n_total = len(y_probs)

sq_cost = (MONTHLY_APPLICANTS * MANUAL_REVIEW_COST) + (MONTHLY_APPLICANTS * 0.04 * 0.10 * FRAUD_LOSS_PER_FN)
ai_review = (yellow_count / n_total) * MONTHLY_APPLICANTS * MANUAL_REVIEW_COST
ai_fraud = (MONTHLY_APPLICANTS * 0.04 * 0.005 * FRAUD_LOSS_PER_FN)
ai_total = ai_review + ai_fraud
monthly_savings = sq_cost - ai_total
annual_savings = monthly_savings * 12

print("""
======================================================================
              EXECUTIVE P&L IMPACT SCORECARD
======================================================================
OPERATIONAL TRIAGE DISTRIBUTION:
  🟢 Tier 1 (Auto-Approve):       {:.1f}%  (Instant 2-sec clearance)
  🟡 Tier 2 (Investigator Review): {:.1f}%  (Routed with Grad-CAM)
  🔴 Tier 3 (Auto-Reject):         {:.1f}%  (Fraud Block & SAR Log)
----------------------------------------------------------------------
FINANCIAL COMPARISON (100,000 Applicants / Month):
  Status Quo (100% Manual Review):   ${:,.0f} / mo
  TrustShield AI (3-Tier Triage):    ${:,.0f} / mo
----------------------------------------------------------------------
  NET MONTHLY VALUE GENERATED:       ${:,.0f} / mo
  PROJECTED ANNUAL VALUE:            ${:,.0f} / year
======================================================================
""".format(
    (green_count / n_total) * 100,
    (yellow_count / n_total) * 100,
    (red_count / n_total) * 100,
    sq_cost,
    ai_total,
    monthly_savings,
    annual_savings
))

## 10. Download Trained Checkpoint for Streamlit Frontend

In [ ]:
from google.colab import files

print("Downloading best_model.pth...")
files.download('output/best_model.pth')

print("Downloading test_predictions.csv...")
files.download('output/test_predictions.csv')
